In [21]:
import pandas as pd
print("pandas loaded")
import os
print(os.getcwd())

pandas loaded
c:\Users\jihad\Documents\MEGA\Classes\CS420\aeromind\research\notebooks


In [22]:
# =========================
# LOAD DATA
# =========================
log_path = "../../data/logs/gesture_research_logs.csv"
df = pd.read_csv(log_path)
print("Data loaded")

Data loaded


In [23]:
# =========================
# CLEAN
# =========================
df = df.fillna("")

for col in ["gesture_true", "gesture_pred", "stable_gesture"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

In [24]:
# =========================
# FILTER DATA
# =========================
gesture_eval = df[df["event_type"] == "gesture_eval"]

valid = {"", "nan", "none", "-", "no_label"}

labeled = gesture_eval[
    (~gesture_eval["gesture_true"].isin(valid)) &
    (~gesture_eval["stable_gesture"].isin(valid))
]


In [25]:
# =========================
# 1. ACCURACY
# =========================
if not labeled.empty:
    accuracy = (labeled["gesture_true"] == labeled["stable_gesture"]).mean()
else:
    accuracy = None

In [26]:
# =========================
# 2. LATENCY
# =========================
for col in ["command_ts_ms", "ack_ts_ms", "e2e_latency_ms"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

api_latency = (df["ack_ts_ms"] - df["command_ts_ms"]).dropna()
e2e_latency = df["e2e_latency_ms"].dropna()

latency_results = {
    "avg_api": api_latency.mean() if not api_latency.empty else None,
    "med_api": api_latency.median() if not api_latency.empty else None,
    "avg_e2e": e2e_latency.mean() if not e2e_latency.empty else None,
    "med_e2e": e2e_latency.median() if not e2e_latency.empty else None,
}

In [27]:
# =========================
# 3. SAFETY METRICS
# =========================
commands = df[df["event_type"].isin(["command_dispatch", "command_blocked"])]

dispatch_count = (commands["event_type"] == "command_dispatch").sum()
blocked_count = (commands["event_type"] == "command_blocked").sum()
total_commands = dispatch_count + blocked_count

if total_commands > 0:
    dispatch_rate = dispatch_count / total_commands
    blocked_rate = blocked_count / total_commands
else:
    dispatch_rate = None
    blocked_rate = None

In [28]:
# =========================
# 4. GESTURE DISTRIBUTION
# =========================
stable_valid = gesture_eval[
    ~gesture_eval["stable_gesture"].isin(valid)
]

gesture_dist = stable_valid["stable_gesture"].value_counts(normalize=True)

In [29]:
# =========================
# PRINT RESULTS (THIS IS WHAT YOU PUT IN PAPER)
# =========================
print("\n===== RESULTS =====\n")

# Accuracy
if accuracy is not None:
    print(f"Accuracy: {accuracy:.2f}")
else:
    print("Accuracy: Not available (no labeled data)")

# Latency
print("\nLatency:")
print(f"Average API Latency: {latency_results['avg_api']:.2f} ms" if latency_results['avg_api'] else "N/A")
print(f"Median API Latency: {latency_results['med_api']:.2f} ms" if latency_results['med_api'] else "N/A")
print(f"Average E2E Latency: {latency_results['avg_e2e']:.2f} ms" if latency_results['avg_e2e'] else "N/A")
print(f"Median E2E Latency: {latency_results['med_e2e']:.2f} ms" if latency_results['med_e2e'] else "N/A")

# Safety
print("\nSafety:")
print(f"Dispatch Rate: {dispatch_rate:.2f}" if dispatch_rate else "N/A")
print(f"Blocked Rate: {blocked_rate:.2f}" if blocked_rate else "N/A")

# Distribution
print("\nTop Gestures:")
print(gesture_dist.head(5))


===== RESULTS =====

Accuracy: Not available (no labeled data)

Latency:
Average API Latency: 10.95 ms
Median API Latency: 5.00 ms
Average E2E Latency: 19634.00 ms
Median E2E Latency: 19634.00 ms

Safety:
Dispatch Rate: 0.40
Blocked Rate: 0.60

Top Gestures:
stable_gesture
fist             0.542650
open_palm        0.398669
point_up         0.026013
l_shape_right    0.012704
thumbs_up        0.010284
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

log_path = "../../data/logs/gesture_research_logs.csv"
df = pd.read_csv(log_path)

# CLEAN
for col in ["event_type", "gesture_true", "stable_gesture"]:
    df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

valid = {"", "nan", "none", "-", "no_label"}

# FILTER LABELED DATA
labeled = df[
    (df["event_type"] == "gesture_eval") &
    (~df["gesture_true"].isin(valid)) &
    (~df["stable_gesture"].isin(valid))
].copy()

# 🔥 CRITICAL CHECK
print("Labeled rows:", len(labeled))

if len(labeled) == 0:
    raise ValueError("No labeled data found → you cannot create Figure 4 from this file")

print("\nActual gestures:")
print(labeled["gesture_true"].value_counts())

print("\nPredicted gestures:")
print(labeled["stable_gesture"].value_counts())

# BUILD CONFUSION MATRIX
cm = pd.crosstab(labeled["gesture_true"], labeled["stable_gesture"])

# NORMALIZE (row-wise)
cm_norm = cm.div(cm.sum(axis=1), axis=0).fillna(0)

# 🔥 CLEAN PLOT (no dark mode garbage)
plt.close("all")
plt.style.use("default")

fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar=True,
    ax=ax
)

ax.set_title("Confusion Matrix of Stabilized Gesture Predictions", fontsize=14)
ax.set_xlabel("Predicted Gesture")
ax.set_ylabel("Actual Gesture")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()

# SAVE CLEAN IMAGE
plt.savefig("../../research/confusion_matrix.png", dpi=300, bbox_inches="tight", facecolor="white")

print("Saved → ../../research/confusion_matrix.png")

plt.show()

gesture_research_logs.csv -> labeled rows: 0
gesture_research_logs.legacy_20260411105617.csv -> labeled rows: 0
run_20260411082348.csv -> labeled rows: 0
run_20260411105617.csv -> labeled rows: 0
run_20260411111955.csv -> labeled rows: 0
run_20260411114617.csv -> labeled rows: 0
